# Word2Vec resources

### Alfio Ferrara


In [1]:
from gensim.models import Word2Vec

## Main functionalities of `gensim` implementation

In [2]:
from nltk.tokenize import word_tokenize
from scipy.spatial import distance
import pandas as pd
import ast
from nltk.tokenize import word_tokenize
from collections import defaultdict

In [3]:
recipe_corpus = []
datafile = "/Users/Flint/Data/recipes/it_recipes.csv"
df = pd.read_csv(datafile, index_col=0)
df = df.dropna()
for i, row in df.iterrows():
    recipe = []
    category = row['Categoria'].lower()
    title = row['Nome'].lower()
    ingredients = [x[0].lower() for x in ast.literal_eval(row['Ingredienti'])]
    text = word_tokenize(row['Steps'].lower().replace("'", " "), language='italian')
    recipe.append(category)
    recipe.append(title)
    recipe.extend(ingredients)
    recipe.extend(text)
    recipe_corpus.append(recipe)

In [4]:
categories = defaultdict(list)
for recipe in recipe_corpus:
    cat = recipe[0]
    categories[cat].append(recipe)
for cat, docs in categories.items():
    print(f"Num di ricette in {cat}: {len(docs)}")

Num di ricette in dolci: 1742
Num di ricette in primi piatti: 1313
Num di ricette in salse e sughi: 91
Num di ricette in lievitati: 329
Num di ricette in secondi piatti: 830
Num di ricette in contorni: 175
Num di ricette in antipasti: 887
Num di ricette in piatti unici: 271
Num di ricette in torte salate: 105
Num di ricette in bevande: 71
Num di ricette in insalate: 57
Num di ricette in marmellate e conserve: 62


In [5]:
print(f"Esempio di dolci: {categories['dolci'][0][:6]}")
print(f"Esempio di primi: {categories['primi piatti'][0][:6]}")

Esempio di dolci: ['dolci', 'tiramisù', 'mascarpone', 'uova', 'savoiardi', 'zucchero']
Esempio di primi: ['primi piatti', 'lasagne alla bolognese', 'semola di grano duro rimacinata', 'farina 00', 'spinaci', 'uova']


In [9]:
recipe_model = Word2Vec(sentences=recipe_corpus, vector_size=50, window=10, 
                        min_count=0, workers=8, epochs=50)

### Similarity

In [28]:
recipe_model.wv.most_similar('frittata')

[('cialda', 0.7408143877983093),
 ('torta', 0.6834738850593567),
 ('polenta', 0.6197603344917297),
 ('crepe', 0.5993174910545349),
 ('composta', 0.5974459648132324),
 ('piadina', 0.5969491600990295),
 ('parmigiana', 0.5922929048538208),
 ('barba', 0.5830387473106384),
 ('crostata', 0.5826994776725769),
 ('besciamella', 0.5769428610801697)]

## Compositionality

In [12]:
dm = recipe_model.wv.doesnt_match(['pasta', 'spaghetti', 'noodles', 'mela'])
common = recipe_model.wv.get_mean_vector(['pasta', 'spaghetti', 'noodles', 'risotto'])
common_word = recipe_model.wv.similar_by_vector(common)
analogy = recipe_model.wv.most_similar(positive=['bollire', 'olio'], negative=['acqua'])

In [13]:
print(f"Doesn't match: {dm}")
print(f"Common terms: {common_word}")
print(f"Analogy: {analogy}")

Doesn't match: mela
Common terms: [('risotto', 0.723035454750061), ('fusilli', 0.7043164968490601), ('rigatoni', 0.6961302757263184), ('vermicelli', 0.6920931935310364), ('spaghetti', 0.6827304363250732), ('bucatini', 0.6499848365783691), ('paccheri', 0.6403021812438965), ('tortiglioni', 0.6249783635139465), ('malloreddus', 0.6189165711402893), ('fregola', 0.6181121468544006)]
Analogy: [('scaldare', 0.8130054473876953), ('appassire', 0.7381464242935181), ('rosolare', 0.7367146611213684), ('cuocere', 0.7195600271224976), ('scottare', 0.7083412408828735), ('soffriggere', 0.7071389555931091), ('stufare', 0.7061246633529663), ('dorare', 0.7024781703948975), ('sfrigolare', 0.6932980418205261), ('passire', 0.6893314719200134)]


## Compare models vectors to measure a shift in meaning

In [14]:
dolci_corpus = categories['dolci']
primi_corpus = categories['primi piatti']
print(f"Dolci: {len(dolci_corpus)}, Primi: {len(primi_corpus)}")

Dolci: 1742, Primi: 1313


### Fine tune the global model to specific sub-corpora

In [15]:
import copy
import numpy as np

In [17]:
generic_corpus = []
for cat, docs in categories.items():
    if cat not in ['primi piatti', 'dolci']:
        generic_corpus.extend(docs)
recipe_model = Word2Vec(sentences=generic_corpus, vector_size=50, window=9, 
                        min_count=0, workers=8, epochs=50)

In [18]:
m_d = copy.deepcopy(recipe_model)
m_p = copy.deepcopy(recipe_model)

In [19]:
m_d.train(dolci_corpus, total_examples=recipe_model.corpus_count, epochs=recipe_model.epochs)
m_p.train(primi_corpus, total_examples=recipe_model.corpus_count, epochs=recipe_model.epochs)

(11342847, 15694800)

In [20]:
word = 'zucchero'
general = dict(recipe_model.wv.most_similar(word))
dol = dict(m_d.wv.most_similar(word))
pri = dict(m_p.wv.most_similar(word))

In [21]:
for k, v in general.items():
    g, d, p = v, dol.get(k, 0), pri.get(k, 0)
    print(f"{k} => {np.round(g, 3)} | D: {np.round(d - g, 3)} | P: {np.round(p - g, 3)}")

sciroppo => 0.821 | D: -0.821 | P: -0.02
smoothie cheesecake => 0.77 | D: -0.037 | P: -0.037
strutto => 0.77 | D: 0.049 | P: -0.024
zafferano => 0.768 | D: 0.01 | P: -0.046
shaker => 0.76 | D: -0.76 | P: -0.027
zenzero => 0.727 | D: -0.031 | P: 0.033
spritz => 0.716 | D: -0.716 | P: 0.001
mele fujion => 0.714 | D: -0.714 | P: -0.714
zabaione => 0.713 | D: -0.713 | P: -0.006
strainer => 0.702 | D: -0.702 | P: -0.702


In [22]:
word = 'zucchero'
v0, vit, vch = recipe_model.wv.get_vector(word), m_d.wv.get_vector(word), m_p.wv.get_vector(word)

In [23]:
print(f"Moving to Dolci: {distance.cosine(vit, v0)}")
print(f"Moving to Primi: {distance.cosine(vch, v0)}")
print(f"Moving from Dolci to Primi: {distance.cosine(vch, vit)}")

Moving to Dolci: 0.14830491317490524
Moving to Primi: 0.04797813738357182
Moving from Dolci to Primi: 0.20225496366077766


## Allenamento di modelli indipendenti

In [24]:
primi_model = Word2Vec(sentences=primi_corpus, vector_size=50, window=9, 
                        min_count=0, workers=8, epochs=50)
dolci_model = Word2Vec(sentences=dolci_corpus, vector_size=50, window=9, 
                        min_count=0, workers=8, epochs=50)

In [27]:
word = "frittata"
primi_model.wv.most_similar(word)

[('capovolgetela', 0.8347384929656982),
 ('scammaro', 0.755085825920105),
 ('6', 0.5986912846565247),
 ('anticipatevi', 0.5887553095817566),
 ('finito', 0.5850023627281189),
 ('alluminio', 0.582649827003479),
 ('7', 0.5710124969482422),
 ('carnevale', 0.569878876209259),
 ('40–50', 0.5655421018600464),
 ('utilizzarli', 0.5595080852508545)]

In [26]:
dolci_model.wv.most_similar(word)

[('miele', 0.5706114768981934),
 ('tortino al cioccolato bianco con panna al maraschino', 0.5698800086975098),
 ('strutto', 0.5598289966583252),
 ('torta al limone senza uova', 0.5555760264396667),
 ('zucchero a velo fatto in casa', 0.525426983833313),
 ('sfouf', 0.5215548276901245),
 ('waffle sandwich alla nutella', 0.5203024744987488),
 ('carnevaleschi', 0.515587568283081),
 ('zafferano', 0.5146864652633667),
 ('pasta frolla', 0.5101369023323059)]

# Progettazione di un'applicazione basata su Word2Vec
- diacronia
- confrontare due edizioni dello stesso testo
- diastratia: estrazione sociale dei parlanti
    - profilazione del parlante
    - studiare la variazione del lessico proveniente da fonti di diversa estrazione
- variazioni lessicali su base tematica
- variazioni lessicale su base geografica
- umano vs AI
- variazione fauna fra fonti enciclopediche

## Strumenti
- corpus
    - reale
    - sintentico
- word2vec
- research question
- protocollo sperimentale
- metriche